# POLITSKY24 Dataset Accessing and Testing

In [4]:
import os
import pandas as pd

from pathlib import Path

# CHANGE THIS TO THE PATH WHERE THE DATASET IS LOCATED
data_path = Path("D:\Thesis-related\POLITISKY24")

<>:7: SyntaxWarning: invalid escape sequence '\T'
<>:7: SyntaxWarning: invalid escape sequence '\T'
C:\Users\andre\AppData\Local\Temp\ipykernel_8660\754992423.py:7: SyntaxWarning: invalid escape sequence '\T'
  data_path = Path("D:\Thesis-related\POLITISKY24")


In [5]:
# Access the parquet files in the data path

parquet_files = list(data_path.glob("*.parquet"))
parquet_files

[WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_full_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_network_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_history_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_list_for_stance_detection.parquet')]

In [6]:
import polars as pl

# Load each dataset manually into individual variables
user_network_df = pl.read_parquet(str(data_path / "user_network_dataset.parquet"))
user_post_history_df = pl.read_parquet(str(data_path / "user_post_history_dataset.parquet"))
user_post_list_df = pl.read_parquet(str(data_path / "user_post_list_for_stance_detection.parquet"))

# Stance label datasets
llm_full_stance_df = pl.read_parquet(str(data_path / "llm_annotated_full_user_stance_dataset.parquet"))
llm_val_stance_df = pl.read_parquet(str(data_path / "llm_annotated_validation_user_stance_dataset.parquet"))
human_val_stance_df = pl.read_parquet(str(data_path / "human_annotated_validation_user_stance_dataset.parquet"))

# Stance relevancy datasets (Note: surround filename strings in quotes)
human_val_relevancy_post_df = pl.read_parquet(
    str(data_path / "human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet")
)
human_val_relevancy_query_df = pl.read_parquet(
    str(data_path / "human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet")
)

## Graph Creation Testing

The code under this section of the notebook is untested and unverified. Double check before being tested.

Simple idea of how we create the data needed for the GNN: User Network Graph creation (likes and reposts) --> Connect this to the LLM full stance using the user id. <br>
    - Verify if they have the same id types (int64) and if they can be connect to each other

In [7]:
import polars as pl

core = user_post_history_df["UserId"].unique()

reposts = (
    user_post_history_df
    .filter(pl.col("IsRepost"))
    .filter(pl.col("SourceUserId").is_in(core)) # keep core users only
    .filter(pl.col("UserId") != pl.col("SourceUserId"))  # drop self-loops
    .group_by("UserId", "SourceUserId")
    .agg(pl.len().alias("weight"))
    .rename({"UserId": "src", "SourceUserId": "dst"})
)

C:\Users\andre\AppData\Local\Temp\ipykernel_8660\2874847685.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("SourceUserId").is_in(core)) # keep core users only


In [8]:
reposts

src,dst,weight
i64,i64,u32
299644,103065,9
387617,240954,2
119244,102757,1
136038,274785,3
195964,212467,1
…,…,…
399897,334118,1
328473,99654,1
287245,302142,1


In [9]:
net = pl.read_parquet(str(data_path / "user_network_dataset.parquet"))  # use the actual filename/format from Zenodo
likes = (
    net.filter(pl.col("InteractionType") == "Like")  # adjust column names to match the file
    .select(src="SourceUserId", dst="TargetUserId", weight="Count")
)

In [10]:
likes

src,dst,weight
i64,i64,i64
9,9,1
9,13873,1
9,14511,1
9,62997,1
9,92773,3
…,…,…
401391,385828,4
401391,392266,1
401391,394655,1


Also do the same for quotes to test all possible interactions (also replies but would have to manually derive the connections in `user_post_history`)

In [11]:
quotes = (
    net.filter(pl.col("InteractionType") == "Quote")
    .select(src="SourceUserId", dst="TargetUserId", weight="Count")
)

In [12]:
quotes

src,dst,weight
i64,i64,i64
9,9,42
9,10,2
9,2549,1
9,3125,1
9,4109,1
…,…,…
401391,384107,3
401391,384773,1
401391,398501,2


In [13]:
replies = (
    user_post_history_df
    .filter(pl.col("IsReply"))
    .filter(pl.col("ParentUserId").is_in(core))
    .filter(pl.col("UserId") != pl.col("ParentUserId"))
    .group_by("UserId", "ParentUserId")
    .agg(pl.len().alias("weight"))
    .rename({"UserId": "src", "ParentUserId": "dst"})
)

C:\Users\andre\AppData\Local\Temp\ipykernel_8660\411258742.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("ParentUserId").is_in(core))


In [14]:
import torch

nodes = core.sort()
idx = {u: i for i, u in enumerate(nodes.to_list())}

def to_edge_index(edges):
    s = [idx[u] for u in edges["src"]]
    d = [idx[u] for u in edges["dst"]]
    return torch.tensor([s, d]), torch.tensor(edges["weight"].to_list(), dtype=torch.float)

edge_index, edge_weight = to_edge_index(likes)  # or reposts

#### Build graph using NetworkX

In [15]:
# NOTE: For now, the graph construction is separate for each interaction type
# We can convert this to a single multi-relational graph later on if needed

In [16]:
import networkx as nx

def build_graph(df: pl.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_weighted_edges_from(
        df.select("src", "dst", "weight").iter_rows(),
        weight="weight"
    )
    return G

In [17]:
reposts_graph = build_graph(reposts)
likes_graph = build_graph(likes)
quotes_graph = build_graph(quotes)
replies_graph = build_graph(replies)

**Attach stance labels**

In [18]:
llm = pl.read_parquet(str(data_path / "llm_annotated_full_user_stance_dataset.parquet"))

In [19]:
print(llm["TargetEntity"].value_counts())
print(llm["StanceLabel"].value_counts())
print(llm["ConfidenceLevel"].value_counts())

shape: (2, 2)
┌──────────────┬───────┐
│ TargetEntity ┆ count │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ Harris       ┆ 8022  │
│ Trump        ┆ 8022  │
└──────────────┴───────┘
shape: (3, 2)
┌─────────────┬───────┐
│ StanceLabel ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ Favor       ┆ 3071  │
│ Against     ┆ 7832  │
│ Neither     ┆ 5141  │
└─────────────┴───────┘
shape: (13, 2)
┌─────────────────┬───────┐
│ ConfidenceLevel ┆ count │
│ ---             ┆ ---   │
│ f64             ┆ u32   │
╞═════════════════╪═══════╡
│ 0.7             ┆ 300   │
│ 0.65            ┆ 2     │
│ 0.75            ┆ 25    │
│ 0.5             ┆ 14    │
│ 0.9             ┆ 2003  │
│ …               ┆ …     │
│ 0.95            ┆ 7274  │
│ 0.99            ┆ 49    │
│ 0.85            ┆ 886   │
│ 0.6             ┆ 61    │
│ null            ┆ 4961  │
└─────────────────┴───────┘


In [20]:
llm_stances = (
    llm.select("UserId", "TargetEntity", "StanceLabel")
    .pivot(
        on="TargetEntity",
        index="UserId",
        values=["StanceLabel"],
        aggregate_function="first"
    )
)
llm_stances.head()

UserId,Harris,Trump
i64,str,str
9,"""Favor""","""Against"""
10,"""Favor""","""Against"""
49,"""Against""","""Against"""
105,"""Neither""","""Against"""
144,"""Against""","""Against"""


In [21]:
# Get the aggregate count of TargetEntities and StanceLabels
aggregated_count_labels = (
    llm.group_by("TargetEntity", "StanceLabel")
    .agg(pl.count("StanceLabel").alias("count"))
    .sort("count", descending=True)
)
aggregated_count_labels


TargetEntity,StanceLabel,count
str,str,u32
"""Trump""","""Against""",6759
"""Harris""","""Neither""",4223
"""Harris""","""Favor""",2726
"""Harris""","""Against""",1073
"""Trump""","""Neither""",918
"""Trump""","""Favor""",345


In [22]:
attrs = {
    r["UserId"]: {"Trump": r["Trump"], "Harris": r["Harris"]}
    for r in llm_stances.iter_rows(named=True)
}

In [23]:
nx.set_node_attributes(reposts_graph, attrs)
nx.set_node_attributes(likes_graph, attrs)
nx.set_node_attributes(quotes_graph, attrs)
nx.set_node_attributes(replies_graph, attrs)

In [24]:
interaction_graphs = [reposts_graph, likes_graph, quotes_graph, replies_graph]
graph_names = ["Reposts", "Likes", "Quotes", "Replies"]
stance_users = set(llm["UserId"].unique()) 

for G in interaction_graphs:
    graph_users = set(G.nodes())

    labeled_nodes = sum(1 for n in G if G.nodes[n].get("Trump") is not None or G.nodes[n].get("Harris") is not None)
    unlabeled_nodes = G.number_of_nodes() - labeled_nodes

    print(f"Graph: {graph_names[interaction_graphs.index(G)]}")
    print("--------------------------------------------------------------")
    print(f"{labeled_nodes}/{G.number_of_nodes()} nodes have a Trump or Harris label") # made this more general since the count before is just the same with both entities
    print(f"{unlabeled_nodes}/{G.number_of_nodes()} nodes are unlabeled") # users in the graph data are not in the stance dataset
    print(f"{len(stance_users - graph_users)} users in the stance dataset are not in the graph data") # users with stances that never interacted with that type of interaction
    print("--------------------------------------------------------------")

Graph: Reposts
--------------------------------------------------------------
7678/8164 nodes have a Trump or Harris label
486/8164 nodes are unlabeled
344 users in the stance dataset are not in the graph data
--------------------------------------------------------------
Graph: Likes
--------------------------------------------------------------
7937/8454 nodes have a Trump or Harris label
517/8454 nodes are unlabeled
85 users in the stance dataset are not in the graph data
--------------------------------------------------------------
Graph: Quotes
--------------------------------------------------------------
6873/7315 nodes have a Trump or Harris label
442/7315 nodes are unlabeled
1149 users in the stance dataset are not in the graph data
--------------------------------------------------------------
Graph: Replies
--------------------------------------------------------------
7768/8241 nodes have a Trump or Harris label
473/8241 nodes are unlabeled
254 users in the stance dataset 

Since there are users that interact with users involved in the POLITISKY24 dataset, we can try to implement a semi-supervised ml model using a GNN to determine the stances of such users based on their structure in the graph. This might also prove the theories for majority illusion and homophily.

In [25]:
llm.schema

Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [26]:
# Check for duplicate stance labels for the same user and target entity
duplicate_count = llm.group_by(["UserId", "TargetEntity"]).len().filter(pl.col("len") > 1).height
print(f"Duplicate stance labels for the same user and target entity: {duplicate_count}")

# Check for missing values in the TargetEntity and StanceLabel columns
missing_count = llm.filter(
    pl.col("TargetEntity").is_null() 
    | pl.col("StanceLabel").is_null()
    | (pl.col("TargetEntity") == "")
    | (pl.col("StanceLabel") == "")
).height

print(f"Rows missing TargetEntity or StanceLabel: {missing_count}")

Duplicate stance labels for the same user and target entity: 0
Rows missing TargetEntity or StanceLabel: 0


## Network Checking for Methodology

In [ ]:
# Check number of nodes and edges for each interaction graph
for G in interaction_graphs:
    graph_name = graph_names[interaction_graphs.index(G)]
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()
    edge_density = nx.density(G)
    edge_density_percent = edge_density * 100


    avg_degree = num_edges / num_nodes 
    reciprocity = nx.reciprocity(G) 
    lwcc = max(nx.weakly_connected_components(G), key=len)
    lwcc_pct = len(lwcc) / num_nodes * 100
    lscc = max(nx.strongly_connected_components(G), key=len)
    lscc_pct = len(lscc) / num_nodes * 100

    
    print(f"""{graph_name} Graph: 
            {num_nodes} nodes, 
            {num_edges} edges, 
            {edge_density_percent:.2f}% edge density, 
            {avg_degree:.2f} average degree, 
            {reciprocity:.4f} reciprocity, 
            {len(lwcc)} nodes in the largest weakly connected component ({lwcc_pct:.2f}% of total nodes), 
            {len(lscc)} nodes in the largest strongly connected component ({lscc_pct:.2f}% of total nodes)""")
    

# Possible discrepancy between number of total number of nodes with lwcc and lscc, 
# as some nodes may not be part of any connected component.

Reposts Graph: 
            8164 nodes, 
            522047 edges, 
            0.78% edge density, 
            63.95 average degree, 
            0.1034 reciprocity, 
            8161 nodes in the largest weakly connected component (99.96% of total nodes), 
            6513 nodes in the largest strongly connected component (79.78% of total nodes)
Likes Graph: 
            8454 nodes, 
            991317 edges, 
            1.39% edge density, 
            117.26 average degree, 
            0.2460 reciprocity, 
            8442 nodes in the largest weakly connected component (99.86% of total nodes), 
            7783 nodes in the largest strongly connected component (92.06% of total nodes)
Quotes Graph: 
            7315 nodes, 
            160498 edges, 
            0.30% edge density, 
            21.94 average degree, 
            0.0999 reciprocity, 
            7184 nodes in the largest weakly connected component (98.21% of total nodes), 
            3463 nodes in the largest st